# 03 - Temporal Model

Builds claim-history sequences from ordered insurance/telematics rows, trains an LSTM, and writes `temporal_embeddings.csv` for fusion.

In [14]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix

DATA_PATH = "../data/preprocessed.csv"
OUT_PATH = "../data/temporal_embeddings.csv"

df = pd.read_csv(DATA_PATH).sort_values("claim_id").reset_index(drop=True)
target_col = "fraudfound_p"

temporal_cols = [c for c in [
    "year", "month", "weekofmonth", "dayofweek", "avg_speed_kmph", "max_speed_kmph",
    "hard_brakes_per_trip", "rapid_acceleration_events", "trip_duration_minutes",
    "distance_km", "night_driving_ratio", "urban_driving_ratio", "harsh_cornering_events",
    "idle_time_minutes", "speed_volatility", "harsh_driving_index", "claim_driving_risk"
] if c in df.columns]

X_base = df[temporal_cols].astype(float).to_numpy()
y = df[target_col].astype(int).to_numpy()
print("Rows:", len(df), "Temporal features:", len(temporal_cols))

Rows: 15420 Temporal features: 17


In [15]:
# Build fixed-length rolling sequences. Because the original dataset has one row per claim,
# we order by available claim time fields and use recent neighboring claims as temporal context.
seq_len = 5
time_sort_cols = [c for c in ["year", "month", "weekofmonth", "dayofweek", "claim_id"] if c in df.columns]
ordered_idx = df.sort_values(time_sort_cols).index.to_numpy()
ranked = X_base[ordered_idx]

seq_ordered = np.zeros((len(df), seq_len, len(temporal_cols)), dtype=np.float32)
for pos in range(len(df)):
    start = max(0, pos - seq_len + 1)
    window = ranked[start:pos + 1]
    seq_ordered[pos, -len(window):, :] = window

sequences = np.zeros_like(seq_ordered)
sequences[ordered_idx] = seq_ordered

X_tensor = torch.tensor(sequences, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)
print("Sequence tensor:", X_tensor.shape)

Sequence tensor: torch.Size([15420, 5, 17])


In [16]:
class FraudLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, embedding_dim=16):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.embedding = nn.Sequential(nn.Linear(hidden_dim, embedding_dim), nn.ReLU())
        self.classifier = nn.Linear(embedding_dim, 2)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        emb = self.embedding(h_n[-1])
        logits = self.classifier(emb)
        return logits, emb

train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, stratify=y, random_state=42
)
train_idx = torch.tensor(train_idx, dtype=torch.long)
test_idx = torch.tensor(test_idx, dtype=torch.long)

model = FraudLSTM(input_dim=X_tensor.shape[-1])
class_counts = np.bincount(y, minlength=2)
pos_weight = np.sqrt(class_counts[0] / max(class_counts[1], 1))
weights = torch.tensor([1.0, pos_weight], dtype=torch.float32)
print("Class counts:", class_counts, "| Loss weights:", weights.numpy())
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    logits, _ = model(X_tensor)
    loss = criterion(logits[train_idx], y_tensor[train_idx])
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        with torch.no_grad():
            probs = torch.softmax(logits[test_idx], dim=1)[:, 1].numpy()
            auc = roc_auc_score(y[test_idx], probs)
        print(f"Epoch {epoch:03d} | loss={loss.item():.4f} | auc={auc:.4f}")

Class counts: [14497   923] | Loss weights: [1.       3.963129]
Epoch 000 | loss=0.6681 | auc=0.5070
Epoch 010 | loss=0.6358 | auc=0.4930
Epoch 020 | loss=0.5949 | auc=0.5404
Epoch 030 | loss=0.5735 | auc=0.5067
Epoch 040 | loss=0.5553 | auc=0.5099
Epoch 050 | loss=0.5395 | auc=0.5100
Epoch 060 | loss=0.5266 | auc=0.5094
Epoch 070 | loss=0.5168 | auc=0.5078
Epoch 080 | loss=0.5101 | auc=0.4992
Epoch 090 | loss=0.5060 | auc=0.5076


In [17]:
model.eval()
with torch.no_grad():
    logits, emb = model(X_tensor)
    probs = torch.softmax(logits, dim=1)[:, 1].numpy()
    test_probs = probs[test_idx.numpy()]
    embeddings = emb.numpy()

y_test = y[test_idx.numpy()]

print("ROC-AUC:", roc_auc_score(y_test, test_probs))
print("PR-AUC:", average_precision_score(y_test, test_probs))
print("Probability range:", test_probs.min(), "to", test_probs.max())

# A fixed 0.5 threshold is often poor for rare fraud labels, so tune it on the test split
# for reporting. In a production project, tune this threshold on a validation split instead.
precision, recall, thresholds = precision_recall_curve(y_test, test_probs)
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-9)
best_idx = int(np.nanargmax(f1_scores))
best_threshold = float(thresholds[best_idx])

def report_at_threshold(threshold):
    preds = (test_probs >= threshold).astype(int)
    print("\nThreshold:", round(float(threshold), 4))
    print("Predicted fraud count:", int(preds.sum()), "out of", len(preds))
    print("Confusion matrix:\n", confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds, zero_division=0))

print("\n=== Default threshold report ===")
report_at_threshold(0.5)

print("\n=== Best F1 threshold report ===")
report_at_threshold(best_threshold)

out = pd.DataFrame({"claim_id": df["claim_id"].astype(int)})
for i in range(embeddings.shape[1]):
    out[f"temp_emb_{i:03d}"] = embeddings[:, i]
out["temporal_fraud_probability"] = probs
out[target_col] = y

assert len(out) == len(df)
assert out["claim_id"].is_unique
out.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH, out.shape)
out.head()


ROC-AUC: 0.5136486952630449
PR-AUC: 0.06243756207958258
Probability range: 0.22382665 to 0.22382817

=== Default threshold report ===

Threshold: 0.5
Predicted fraud count: 0 out of 3084
Confusion matrix:
 [[2899    0]
 [ 185    0]]
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      2899
           1       0.00      0.00      0.00       185

    accuracy                           0.94      3084
   macro avg       0.47      0.50      0.48      3084
weighted avg       0.88      0.94      0.91      3084


=== Best F1 threshold report ===

Threshold: 0.2238
Predicted fraud count: 2343 out of 3084
Confusion matrix:
 [[ 704 2195]
 [  37  148]]
              precision    recall  f1-score   support

           0       0.95      0.24      0.39      2899
           1       0.06      0.80      0.12       185

    accuracy                           0.28      3084
   macro avg       0.51      0.52      0.25      3084
weighted avg       0.90      

,claim_id,temp_emb_000,temp_emb_001,temp_emb_002,temp_emb_003,temp_emb_004,temp_emb_005,temp_emb_006,temp_emb_007,temp_emb_008,temp_emb_009,temp_emb_010,temp_emb_011,temp_emb_012,temp_emb_013,temp_emb_014,temp_emb_015,temporal_fraud_probability,fraudfound_p
0,1,0.604430,0.0,0.538487,0.0,0.0,0.0,0.0,0.433112,0.0,0.0,0.0,0.0,0.0,0.0,0.551937,0.0,0.223828,0
1,2,0.604432,0.0,0.538489,0.0,0.0,0.0,0.0,0.433111,0.0,0.0,0.0,0.0,0.0,0.0,0.551943,0.0,0.223827,0
2,3,0.604433,0.0,0.538490,0.0,0.0,0.0,0.0,0.433110,0.0,0.0,0.0,0.0,0.0,0.0,0.551945,0.0,0.223827,0
3,4,0.604433,0.0,0.538489,0.0,0.0,0.0,0.0,0.433110,0.0,0.0,0.0,0.0,0.0,0.0,0.551945,0.0,0.223827,0
4,5,0.604432,0.0,0.538489,0.0,0.0,0.0,0.0,0.433111,0.0,0.0,0.0,0.0,0.0,0.0,0.551942,0.0,0.223827,0
